In [3]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
df_alunos = spark.read.parquet(
    "Files/bronze/alunos/alunos.parquet"
)

display(df_alunos)

StatementMeta(, b47e6e41-e4d8-45f6-82a7-8e4735747437, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d4a6a44c-a356-44ae-bd9e-0d36f3baa168)

In [4]:
df_alunos.printSchema()

StatementMeta(, b47e6e41-e4d8-45f6-82a7-8e4735747437, 6, Finished, Available, Finished, False)

root
 |-- ano: long (nullable = true)
 |-- id_municipio: string (nullable = true)
 |-- id_escola: string (nullable = true)
 |-- id_aluno: string (nullable = true)
 |-- caderno: string (nullable = true)
 |-- serie: string (nullable = true)
 |-- rede: string (nullable = true)
 |-- presenca: string (nullable = true)
 |-- preenchimento_caderno: string (nullable = true)
 |-- alfabetizado: string (nullable = true)
 |-- proficiencia: double (nullable = true)
 |-- peso_aluno: double (nullable = true)



In [3]:
display(df_alunos.limit(100))

StatementMeta(, e3642901-2c27-426a-b1b1-75cd836c8f00, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b3b6b308-85d8-44a0-9859-6d9e18aafbee)

In [5]:
df_alunos_2023 = df_alunos.filter(df_alunos.ano == 2023)

df_alunos_2023.select(
    "id_escola",
    "id_municipio"
).distinct().show(20, truncate=False)

StatementMeta(, b47e6e41-e4d8-45f6-82a7-8e4735747437, 7, Finished, Available, Finished, False)

+---------+------------+
|id_escola|id_municipio|
+---------+------------+
|60035887 |4202909     |
|60020881 |3145604     |
|60038378 |4314407     |
|60000396 |1100098     |
|60003575 |1600303     |
|60038350 |4314407     |
|60019210 |3135209     |
|60038832 |4316808     |
|60018854 |3122306     |
|60023026 |3205002     |
|60038953 |4317202     |
|60020250 |3134004     |
|60035018 |4206207     |
|60039081 |4311759     |
|60038668 |4315305     |
|60022917 |3204559     |
|60000105 |1100122     |
|60039029 |4317608     |
|60038599 |4314902     |
|60039705 |4303004     |
+---------+------------+
only showing top 20 rows



In [6]:
df_alunos_2023.select("id_escola").distinct().count()

StatementMeta(, b47e6e41-e4d8-45f6-82a7-8e4735747437, 8, Finished, Available, Finished, False)

36776

In [1]:

from pyspark.sql import functions as F

# Ajustar o caminho conforme o Lakehouse
caminho = "Files/bronze/censo_escolar/2023/microdados_ed_basica_2023.csv"


df_censo = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("encoding", "ISO-8859-1")
    .option("inferSchema", True)
    .csv(caminho)
)

# Apenas escolas em funcionamento
df_escolas = df_censo.filter(
    F.col("TP_SITUACAO_FUNCIONAMENTO") == 1
)

# Primeira agregação municipal
df_municipios = (
    df_escolas
    .groupBy("NU_ANO_CENSO", "CO_MUNICIPIO")
    .agg(
        F.countDistinct("CO_ENTIDADE").alias("qtd_escolas"),
        F.sum(F.when(F.col("TP_LOCALIZACAO") == 2, 1)
              .otherwise(0)).alias("qtd_escolas_rurais"),
        F.sum(F.when(F.col("IN_INTERNET") == 1, 1)
              .otherwise(0)).alias("qtd_escolas_internet"),
        F.sum("QT_MAT_FUND_AI").alias("matriculas_anos_iniciais")
    )
)

display(df_municipios.limit(10))


StatementMeta(, f0e18e96-09a6-4983-93b2-46d5b4a118ee, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7ad83111-0812-4f96-ad98-03602d79c225)

In [2]:

print("Escolas em funcionamento:", df_escolas.count())

print("Municípios agregados:", df_municipios.count())

print("Chaves duplicadas:",
    df_municipios.groupBy(
        "NU_ANO_CENSO", "CO_MUNICIPIO"
    ).count().filter("count > 1").count()
)

print("Anos encontrados:")
df_municipios.groupBy("NU_ANO_CENSO").count().show()


StatementMeta(, f0e18e96-09a6-4983-93b2-46d5b4a118ee, 4, Finished, Available, Finished, False)

Escolas em funcionamento: 180230
Municípios agregados: 5570
Chaves duplicadas: 0
Anos encontrados:
+------------+-----+
|NU_ANO_CENSO|count|
+------------+-----+
|        2023| 5570|
+------------+-----+



In [5]:
df_bronze_alunos = spark.read.parquet("Files/bronze/alunos/alunos.parquet")

# Municípios distintos da base de alunos em 2023
df_municipios_alunos = (
    df_bronze_alunos.filter(F.col("ano") == 2023)
      .select(
          F.col("id_municipio").cast("string").alias("id_municipio")
      )
      .distinct()
)

# Municípios agregados do Censo Escolar
df_municipios_censo = (
    df_municipios
    .filter(F.col("NU_ANO_CENSO") == 2023)
    .select(
        F.col("CO_MUNICIPIO").cast("string").alias("id_municipio")
    )
    .distinct()
)

# Municípios sem correspondência
df_sem_match = df_municipios_alunos.join(
    df_municipios_censo,
    on="id_municipio",
    how="left_anti"
)

print("Municípios na base de alunos:",
      df_municipios_alunos.count())

print("Municípios sem correspondência:",
      df_sem_match.count())

display(df_sem_match.limit(20))


StatementMeta(, f0e18e96-09a6-4983-93b2-46d5b4a118ee, 7, Finished, Available, Finished, False)

Municípios na base de alunos: 4873
Municípios sem correspondência: 0


SynapseWidget(Synapse.DataFrame, 01d76c47-0124-4922-81b2-ba94ccbac2e9)

In [6]:

colunas = [
    "IN_INTERNET",
    "IN_BIBLIOTECA",
    "IN_SALA_LEITURA",
    "QT_MAT_FUND_AI",
    "QT_DOC_FUND_AI",
    "QT_TUR_FUND_AI",
    "QT_MAT_FUND_AI_INT"
]

for coluna in colunas:
    print(f"{coluna}:")
    df_escolas.groupBy(coluna).count().orderBy(
        F.desc("count")
    ).show(5)


StatementMeta(, f0e18e96-09a6-4983-93b2-46d5b4a118ee, 8, Finished, Available, Finished, False)

IN_INTERNET:
+-----------+------+
|IN_INTERNET| count|
+-----------+------+
|          1|163855|
|          0| 16375|
+-----------+------+

IN_BIBLIOTECA:
+-------------+------+
|IN_BIBLIOTECA| count|
+-------------+------+
|            0|114348|
|            1| 65882|
+-------------+------+

IN_SALA_LEITURA:
+---------------+------+
|IN_SALA_LEITURA| count|
+---------------+------+
|              0|131108|
|              1| 49122|
+---------------+------+

QT_MAT_FUND_AI:
+--------------+-----+
|QT_MAT_FUND_AI|count|
+--------------+-----+
|             0|74691|
|          NULL| 1754|
|             9| 1368|
|            11| 1343|
|             8| 1307|
+--------------+-----+
only showing top 5 rows

QT_DOC_FUND_AI:
+--------------+-----+
|QT_DOC_FUND_AI|count|
+--------------+-----+
|             0|88623|
|             1|12219|
|             5| 6932|
|             7| 6439|
|             6| 6237|
+--------------+-----+
only showing top 5 rows

QT_TUR_FUND_AI:
+--------------+-----+
|QT

In [7]:

from pyspark.sql import functions as F

# Escolas em funcionamento com matrículas nos anos iniciais
df_escolas_ai = df_escolas.filter(
    F.col("QT_MAT_FUND_AI") > 0
)

# Biblioteca OU sala de leitura
df_escolas_ai = df_escolas_ai.withColumn(
    "possui_espaco_leitura",
    F.when(
        (F.col("IN_BIBLIOTECA") == 1) |
        (F.col("IN_SALA_LEITURA") == 1),
        1
    ).otherwise(0)
)

# Agregação municipal
df_silver_censo_2023 = (
    df_escolas_ai
    .groupBy("NU_ANO_CENSO", "CO_MUNICIPIO")
    .agg(
        F.countDistinct("CO_ENTIDADE").alias("qtd_escolas_ai"),
        F.sum(F.when(F.col("TP_LOCALIZACAO") == 2, 1)
              .otherwise(0)).alias("qtd_escolas_rurais"),
        F.sum("IN_INTERNET").alias("qtd_escolas_internet"),
        F.sum("possui_espaco_leitura").alias("qtd_escolas_leitura"),
        F.sum("QT_MAT_FUND_AI").alias("qtd_matriculas_ai"),
        F.sum("QT_DOC_FUND_AI").alias("qtd_docentes_ai"),
        F.sum("QT_TUR_FUND_AI").alias("qtd_turmas_ai"),
        F.sum("QT_MAT_FUND_AI_INT").alias("qtd_matriculas_integral_ai")
    )
    .withColumn(
        "pct_escolas_rurais",
        F.round(F.col("qtd_escolas_rurais") /
                F.col("qtd_escolas_ai") * 100, 2)
    )
    .withColumn(
        "pct_escolas_internet",
        F.round(F.col("qtd_escolas_internet") /
                F.col("qtd_escolas_ai") * 100, 2)
    )
    .withColumn(
        "pct_escolas_leitura",
        F.round(F.col("qtd_escolas_leitura") /
                F.col("qtd_escolas_ai") * 100, 2)
    )
    .withColumn(
        "alunos_por_docente",
        F.when(
            F.col("qtd_docentes_ai") > 0,
            F.round(
                F.col("qtd_matriculas_ai") /
                F.col("qtd_docentes_ai"), 2
            )
        )
    )
    .withColumn(
        "alunos_por_turma",
        F.when(
            F.col("qtd_turmas_ai") > 0,
            F.round(
                F.col("qtd_matriculas_ai") /
                F.col("qtd_turmas_ai"), 2
            )
        )
    )
    .withColumn(
        "pct_matriculas_integral",
        F.round(
            F.col("qtd_matriculas_integral_ai") /
            F.col("qtd_matriculas_ai") * 100, 2
        )
    )
)

display(df_silver_censo_2023.limit(10))


StatementMeta(, f0e18e96-09a6-4983-93b2-46d5b4a118ee, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 42dabbba-f0a4-4403-9940-7fceb3b0f250)

In [8]:

print("Escolas com anos iniciais:", df_escolas_ai.count())
print("Municípios agregados:", df_silver_censo_2023.count())

print("Municípios sem correspondência na base de alunos:",
    df_municipios_alunos.join(
        df_silver_censo_2023.select(
            F.col("CO_MUNICIPIO").cast("string")
             .alias("id_municipio")
        ),
        on="id_municipio",
        how="left_anti"
    ).count()
)


StatementMeta(, f0e18e96-09a6-4983-93b2-46d5b4a118ee, 10, Finished, Available, Finished, False)

Escolas com anos iniciais: 103785
Municípios agregados: 5570
Municípios sem correspondência na base de alunos: 0


In [9]:

indicadores = [
    "pct_escolas_rurais",
    "pct_escolas_internet",
    "pct_escolas_leitura",
    "pct_matriculas_integral"
]

for coluna in indicadores:
    invalidos = df_silver_censo_2023.filter(
        (F.col(coluna) < 0) |
        (F.col(coluna) > 100) |
        F.col(coluna).isNull()
    ).count()
    print(f"{coluna}: {invalidos} inválidos")

print(
    "Municípios com docentes zerados:",
    df_silver_censo_2023.filter(
        F.col("qtd_docentes_ai") == 0
    ).count()
)

print(
    "Municípios com turmas zeradas:",
    df_silver_censo_2023.filter(
        F.col("qtd_turmas_ai") == 0
    ).count()
)


StatementMeta(, f0e18e96-09a6-4983-93b2-46d5b4a118ee, 11, Finished, Available, Finished, False)

pct_escolas_rurais: 0 inválidos
pct_escolas_internet: 0 inválidos
pct_escolas_leitura: 0 inválidos
pct_matriculas_integral: 0 inválidos
Municípios com docentes zerados: 0
Municípios com turmas zeradas: 0


In [10]:

(
    df_silver_censo_2023
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver_censo_escolar_municipal_2023")
)

print("Silver Censo Escolar 2023 salva com sucesso!")


StatementMeta(, f0e18e96-09a6-4983-93b2-46d5b4a118ee, 12, Finished, Available, Finished, False)

Silver Censo Escolar 2023 salva com sucesso!


In [11]:

df_validacao = spark.table(
    "silver_censo_escolar_municipal_2023"
)

print("Registros salvos:", df_validacao.count())
display(df_validacao.limit(10))


StatementMeta(, f0e18e96-09a6-4983-93b2-46d5b4a118ee, 13, Finished, Available, Finished, False)

Registros salvos: 5570


SynapseWidget(Synapse.DataFrame, ea5d45b4-6540-407d-a260-ab4a076930a3)